# Pauli exponential with constant depth

## Implementation of Fig. 7 of https://arxiv.org/pdf/2603.17774

In [1]:
from guppyalgos.primitives.pauli.pauli_exp.pauli_exp_depth1 import pauli_exp_depth1
from guppyalgos.utils.guppy.gates import transversal
from guppylang.std.quantum import qubit, rz

from guppylang import guppy, comptime
from guppylang.std.builtins import array, comptime, result
from guppylang.std.quantum import qubit, cx, h, rz, measure_array, discard_array, measure, discard, x, z

from guppylang.std.debug import state_result
from guppylang.std.angles import angle

from selene_sim import QuantumReplay, Quest, build, MetricStore, Coinflip

import zixy.qubit.pauli as zqp

import numpy as np

Specify the number of qubits (Pauli length), Pauli string, and angle theta

In [2]:
# n_state_qubits = 3
# paulis = zqp.String.from_str("Y0 X1 Z2 ", n_state_qubits)
# n_state_qubits = 2
# paulis = zqp.String.from_str("X0 Y1", n_state_qubits)
n_state_qubits = 2
paulis = zqp.String.from_str("I0 X1", n_state_qubits)


theta = 0.3

The expected state from acting the Pauli exponential operator on the state |0>

In [3]:
from scipy.linalg import expm

pauli_mat = zqp.RealTermSum.from_str(str(paulis), n_state_qubits).to_sparse_matrix(True).todense()
print(f"pauli_mat: {pauli_mat}")
u_mat = expm(1j * (-0.5* np.pi *(theta)) * pauli_mat) #note sign change
state0 = (1/np.sqrt(2**n_state_qubits)) * np.ones(2**n_state_qubits)


sf = u_mat @ state0

for i in range(len(sf)):
    print(f"{sf[i]:.4f} ")

pauli_mat: [[0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]]
0.4455-0.2270j 
0.4455-0.2270j 
0.4455-0.2270j 
0.4455-0.2270j 


Constant depth circuit implementation

In [4]:
from guppyalgos.utils import qarray

rz_method = rz
pauli_gadget = pauli_exp_depth1(paulis, n_state_qubits, rz_method)

@guppy
def main() -> None:
    state_qreg = qarray(comptime(n_state_qubits))
    transversal(h, state_qreg)
    pauli_gadget(state_qreg, angle(comptime(theta)))
    state_result("result_state", state_qreg)
    discard_array(state_qreg)



There are $2^n$ possible midcircuit measurement outcome, where $n$ is the Pauli length. 
Whatever the outcome is, the resulting states should all be the same and match the matrix-vector result

In [5]:
import itertools
combinations = list(itertools.product([True, False], repeat=n_state_qubits))
anc_measurements = [list(comb) for comb in combinations]
partial_replay_measurements = anc_measurements
simulator = QuantumReplay(
    simulator=Quest(random_seed=17),
    resume_with_measurement=True,
    measurements=partial_replay_measurements,
)
em_result = (
    main.emulator(n_state_qubits+n_state_qubits).with_simulator(simulator).with_shots(2**n_state_qubits).run()
)

sv = {}
for i, shot_result in enumerate(em_result.results):
    states = Quest.extract_states_dict(shot_result)
    sv[i] = states["result_state"].get_single_state()
    print(sv[i])



[0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]
[0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]
[0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]
[0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]


In [6]:
u_mat@state0

array([0.44550326-0.22699525j, 0.44550326-0.22699525j,
       0.44550326-0.22699525j, 0.44550326-0.22699525j])